In [17]:
import numpy as np 
import pandas as pd 
import re 
from transformers import AutoTokenizer, AutoModel
from sklearn.preprocessing import LabelEncoder
import torch


In [3]:
newsgrp = pd.read_csv('np20ng.csv')
newsgrp

,source,category,heading,content
0,Gorkhapatra,Education,"अटेर गर्दै कतिपय विद्यालय, चैतमै भर्ना अभियान ...","काठमाडौं,चैत्र ७ गते । उपत्याकाका कतिपय विद्या..."
1,Gorkhapatra,Education,"निर्वाचन प्रचारमा विद्यालय, शिक्षक र विद्यार्...","प्रकाश सिलवाल काठमाडौँ, चैत ६ गते । विगतका निर..."
2,Gorkhapatra,Education,विद्यार्थीलाई परम्परागत सीप सिकाउँदै विद्यालय,"अमरराज नहर्की तनहुँ, चैत्र ५ गते । तनहुँको एक ..."
3,Gorkhapatra,Education,आन्दोलन नगर्न शिक्षामन्त्रीको आग्रह,"गोरखापत्र समाचारदाता काठमाडौँ, चैत ३ गते । शिक..."
4,Gorkhapatra,Education,विश्वविद्यालयमा हडताल नगर्न शिक्षामन्त्रीको आग्रह,"काठमाडौं, चैत २ गते। शिक्षा विज्ञान तथा प्रविध..."
...,...,...,...,...
231211,Aarthiknews,Business,अपर नौगाड जलविद्युत् उत्पादनको अन्तिम तयारी,\nदार्चुला । नौगाड गाउँपालिकामा निर्माणाधीन ...
231212,Aarthiknews,Business,कुलेखानी तेस्रो र माथिल्लो त्रिशुली थ्री ए निर...,\nलामो समयसम्म समस्यामा फसेका नेपाल विद्य...
231213,Aarthiknews,Business,माथिल्लो तामाकोशी लागत बढ्नुमा सञ्चय कोषको हात...,"काठमाडौं, पुस १९ । कर्मचारी सञ्चय कोषले साहू म..."
231214,Aarthiknews,Business,नेपाल हाइड्रो डेभलोपरले लाभांश दिने,"काठमाडौं, पुस १० । नेपाल हाइड्रो डेभलोपर लिमिट..."


In [4]:

len(newsgrp['content'][1])

4536

In [5]:
# we dotn need sorces so dropiing sources 
newsgrp.drop(['source'],axis=1,inplace=True)

In [6]:
newsgrp['content'][3]

'गोरखापत्र समाचारदाता काठमाडौँ, चैत ३ गते । शिक्षा विज्ञान तथा प्रविधिमन्त्री देवेन्द्र पौडेलले विश्वविद्यालयमा बन्द, हडताल तथा आन्दोलनका गतिविधि नगर्न प्राध्यापकहरूलाई आग्रह गर्नुभएको छ । नेपाल प्राध्यापक सङ्घका पदाधिकारीसँग बुधबार मन्त्रालयमा छलफल गर्दै मन्त्री पौडेलले ससाना कुरालाई लिएर विश्वविद्यालयमा हडताल गर्नु गलत भएको बताउनुभयो । उहाँले भन्नुभयो, “लोकतान्त्रिक मर्यादाअनुरूप माग प्रस्तुत गर्न पाइन्छ तर लामो समयसम्म पढाइ अवरुद्ध हुने गरी विश्वविद्यालयमा गरिएको हडताल रोक्नु प-यो ।” लामो समयदेखि आंशिक प्राध्यापक तथा विभिन्न विद्यार्थी सङ्गठनको आन्दोलनका कारण पछिल्लो समय त्रिभुवन विश्वविद्यालय ठप्प छ । सबै विश्वविद्यालयका सेवा आयोगहरूमा एकरूपता कायम गरिने पौडेलले बताउनुभयो । प्राध्यापकका सेवासुविधा सम्बन्धमा सबै राष्ट्रसेवकलाई एउटै मापदण्ड बनाएर अघि बढ्ने योजनामा रहेको मन्त्री पौडेलले सो अवसरमा जानकारी गराउनुभयो । विश्वविद्यालयहरूले समस्या परेको बेला मन्त्रालयमा आउने तर अरू जिम्मेवारी, अनुशासन र नियमन सुधारका अवस्थामा स्वायत्त भएको भन्ने गरिएकोप्रति मन्त्री पौडेलले गुनासो गर्नुभयो ।

In [7]:
newsgrp.nunique()

category        20
heading     227162
content     230964
dtype: int64

In [8]:
# mergre title and 
newsgrp['content_merged'] = newsgrp['heading'].str.cat(newsgrp['content'])
newsgrp.drop(['content','heading'],axis=1,inplace=True)

In [9]:
newsgrp['content_merged'] = newsgrp['content_merged'].apply(lambda x: re.sub(r'[\r\n]+', '', str(x))) #str(x) ensures you dont hit errors if there are NaN or non-string entries.

In [10]:
newsgrp


,category,content_merged
0,Education,"अटेर गर्दै कतिपय विद्यालय, चैतमै भर्ना अभियान ..."
1,Education,"निर्वाचन प्रचारमा विद्यालय, शिक्षक र विद्यार्..."
2,Education,विद्यार्थीलाई परम्परागत सीप सिकाउँदै विद्यालयअ...
3,Education,आन्दोलन नगर्न शिक्षामन्त्रीको आग्रहगोरखापत्र स...
4,Education,विश्वविद्यालयमा हडताल नगर्न शिक्षामन्त्रीको आग...
...,...,...
231211,Business,अपर नौगाड जलविद्युत् उत्पादनको अन्तिम तयारी दा...
231212,Business,कुलेखानी तेस्रो र माथिल्लो त्रिशुली थ्री ए निर...
231213,Business,माथिल्लो तामाकोशी लागत बढ्नुमा सञ्चय कोषको हात...
231214,Business,"नेपाल हाइड्रो डेभलोपरले लाभांश दिनेकाठमाडौं, प..."


In [11]:
# converting labels innto integers 
label = LabelEncoder()
newsgrp['category'] = label.fit_transform(newsgrp['category'])

In [12]:
# save the label encoder for further use 
import pickle
with open('label_encoder.pkl',"wb") as f:
    pickle.dump(label,f)

In [13]:
newsgrp

,category,content_merged
0,6,"अटेर गर्दै कतिपय विद्यालय, चैतमै भर्ना अभियान ..."
1,6,"निर्वाचन प्रचारमा विद्यालय, शिक्षक र विद्यार्..."
2,6,विद्यार्थीलाई परम्परागत सीप सिकाउँदै विद्यालयअ...
3,6,आन्दोलन नगर्न शिक्षामन्त्रीको आग्रहगोरखापत्र स...
4,6,विश्वविद्यालयमा हडताल नगर्न शिक्षामन्त्रीको आग...
...,...,...
231211,3,अपर नौगाड जलविद्युत् उत्पादनको अन्तिम तयारी दा...
231212,3,कुलेखानी तेस्रो र माथिल्लो त्रिशुली थ्री ए निर...
231213,3,माथिल्लो तामाकोशी लागत बढ्नुमा सञ्चय कोषको हात...
231214,3,"नेपाल हाइड्रो डेभलोपरले लाभांश दिनेकाठमाडौं, प..."


In [14]:
with open('stopwords.txt') as f:
    nepali_stop_words = set(f.read().split('\n'))

In [18]:
# tokenizer
tokenizer = AutoTokenizer.from_pretrained('xlm-roberta-base')
model = AutoModel.from_pretrained('xlm-roberta-base')

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

# convert texts to list
texts = newsgrp['content_merged'].tolist()
labels = newsgrp['category'].values

# process in batches
batch_size = 32
embeddings = []

for i in range(0, len(texts), batch_size):
    batch_texts = texts[i:i+batch_size]
    
    # tokenize batch
    inputs = tokenizer(batch_texts, return_tensors='pt', truncation=True, max_length=512, padding=True)
    
    # get embeddings
    with torch.no_grad():
        outputs = model(**inputs)
    
    # save [CLs] token (sentence embeddings)
    batch_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
    embeddings.append(batch_embeddings)
    
    print(f"Processed {i+batch_size}/{len(texts)}")

# Combine all embeddings
import numpy as np
all_embeddings = np.vstack(embeddings)

Processed 32/231216
Processed 64/231216
Processed 96/231216
Processed 128/231216
Processed 160/231216
Processed 192/231216
Processed 224/231216
Processed 256/231216
Processed 288/231216
Processed 320/231216
Processed 352/231216
Processed 384/231216
Processed 416/231216
Processed 448/231216


In [ ]:
tokenizer.save_pretrained('./tokenizer')
